In [0]:
path = '/Workspace/Users/sateeshfrnd@gmail.com/SAMBA/datareadiness_assesment_p/sample_datasets/TotalSummaryDatawithFormulationwithRPM1022and1625.csv' 

In [0]:
   
df = spark.read.csv(path, header=True, inferSchema=True) 
df.printSchema()

root
 |-- TrialBusinessId_ of Sigma: string (nullable = true)
 |-- Mazzoni 1- Sigma Mixer current_mean: double (nullable = true)
 |-- Mazzoni 1- Sigma Mixer current_sum: double (nullable = true)
 |-- Mazzoni 1- Sigma Mixer current_median: double (nullable = true)
 |-- Mazzoni 1- Sigma Mixer current_std: double (nullable = true)
 |-- Mazzoni 1- Plodder vaccum pump Start/stop Indication_mean10to22: double (nullable = true)
 |-- Mazzoni 1- Vaccum in final plodder hopper_mean10to22: double (nullable = true)
 |-- Mazzoni 1 - Billet Temperature at Plodder exit_std10to22: double (nullable = true)
 |-- Mazzoni 1 - Flowrate throughput_std10to22: double (nullable = true)
 |-- Mazzoni 1- Final Plodder Worm motor current_std10to22: double (nullable = true)
 |-- Mazzoni 1- Final plodder Chilled Water flow_std10to22: double (nullable = true)
 |-- Mazzoni 1- Final plodder Chilled water SV on/off indication_std10to22: integer (nullable = true)
 |-- Mazzoni 1- Final plodder Cone Heater temp_std10to22: 

In [0]:
df.show(5,False)

+-------------------------+-----------------------------------+----------------------------------+-------------------------------------+----------------------------------+---------------------------------------------------------------+----------------------------------------------------+--------------------------------------------------------+-----------------------------------------+-----------------------------------------------------+-----------------------------------------------------+---------------------------------------------------------------------+---------------------------------------------------+---------------------------------------------------------+---------------------------------------------------------+------------------------------------------------------------+-------------------------------------------------------------------+-------------------------------------------------+-------------------------------------------+--------------------------------------------

# Configuration

In [0]:
import json
import numpy as np
import pandas as pd

from scipy.stats import chi2, mannwhitneyu, chi2_contingency
# from statsmodels.stats.multitest import multipletests

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

from pyspark.sql import functions as F
from pyspark.sql.types import (
    NumericType, DoubleType, FloatType
)

# Your four observed Sigma Mixer measurements
TARGET_COLUMNS = [
    "Mazzoni 1- Sigma Mixer current_mean",
    "Mazzoni 1- Sigma Mixer current_sum",
    "Mazzoni 1- Sigma Mixer current_median",
    "Mazzoni 1- Sigma Mixer current_std",
]

MAX_SAMPLE_ROWS = 50_000
MISSING_THRESHOLD_PCT = 1.0
MCAR_ALPHA = 0.05
MAR_AUC_THRESHOLD = 0.65
TARGET_ASSOC_ALPHA = 0.01

# Identifier should not be used as a model predictor.
ID_COLUMNS = ["TrialBusinessId_ of Sigma"]

# Prepare the sample and missingness profile

In [0]:
def prepare_miss02_data(df, max_rows=MAX_SAMPLE_ROWS):
    total_rows = df.count()

    if total_rows == 0:
        raise ValueError("Input DataFrame contains no rows.")

    numeric_columns = [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, NumericType)
    ]

    # Handle NaN as missing for floating-point features.
    missing_exprs = []
    for c in df.columns:
        dtype = df.schema[c].dataType

        is_missing = F.col(c).isNull()

        if isinstance(dtype, (DoubleType, FloatType)):
            is_missing = is_missing | F.isnan(F.col(c))

        missing_exprs.append(
            F.sum(F.when(is_missing, 1).otherwise(0)).alias(c)
        )

    counts = df.agg(*missing_exprs).first().asDict()

    profile = pd.DataFrame([
        {
            "feature": c,
            "missing_k": int(counts[c] or 0),
            "missing_pct": 100.0 * (counts[c] or 0) / total_rows
        }
        for c in df.columns
    ])

    eligible = profile.loc[
        profile["missing_pct"] >= MISSING_THRESHOLD_PCT,
        "feature"
    ].tolist()

    # Only sample if necessary. Prefer an existing MISS-01
    # stratified sample if your pipeline already produces one.
    if total_rows > max_rows:
        sampled_df = df.orderBy(F.rand(seed=42)).limit(max_rows)
    else:
        sampled_df = df

    pdf = sampled_df.select(*df.columns).toPandas()

    # Normalize NaN and infinite numeric values.
    for c in numeric_columns:
        pdf[c] = pd.to_numeric(pdf[c], errors="coerce")
        pdf[c] = pdf[c].replace([np.inf, -np.inf], np.nan)

    return {
        "pdf": pdf,
        "profile": profile,
        "numeric_columns": numeric_columns,
        "eligible_columns": eligible,
        "total_rows": total_rows,
        "sample_rows": len(pdf)
    }

In [0]:
prepared = prepare_miss02_data(df)

pdf = prepared["pdf"]
profile = prepared["profile"]
numeric_columns = prepared["numeric_columns"]
eligible_columns = prepared["eligible_columns"]

display(profile.sort_values("missing_pct", ascending=False))

print("Rows:", prepared["total_rows"])
print("Sample rows:", prepared["sample_rows"])
print("Eligible features:", len(eligible_columns))

feature,missing_k,missing_pct
Mazzoni 1- Final Plodder Worm motor current_min10to22,6,4.054054054054054
Mazzoni 1- Final Plodder Worm motor current_max10to22,6,4.054054054054054
Mazzoni 1- Final Plodder Worm motor current_std10to22,6,4.054054054054054
Mazzoni 1- Final plodder worm RPM_min10to22,5,3.3783783783783785
Mazzoni 1- Final plodder Chilled Water flow_min10to22,5,3.3783783783783785
Mazzoni 1- Final plodder Chilled water SV on/off indication_min10to22,5,3.3783783783783785
Mazzoni 1- Final plodder Cone Heater temp_min10to22,5,3.3783783783783785
Mazzoni 1- Final plodder I/L cooling water temp_min10to22,5,3.3783783783783785
Mazzoni 1- Final plodder O/L cooling water temp_min10to22,5,3.3783783783783785
Mazzoni 1- Final plodder Temperature at Cone Inlet_min10to22,5,3.3783783783783785


Rows: 148
Sample rows: 148
Eligible features: 39


**Sampling note:** The random sample above is a bounded fallback, not a stratified sample. For your current 148-row dataset, the complete DataFrame is used. If you later run this on a larger dataset, prefer the stratified sample already generated by your pipeline.

# Little's MCAR test with corrected degrees of freedom

In [0]:
def littles_mcar_test(pdf, numeric_columns, max_iter=200, tol=1e-6):
    # Little's test: numeric features with at least one missing value.
    cols = [
        c for c in numeric_columns
        if c in pdf.columns and pdf[c].isna().any()
    ]

    x = pdf[cols].replace([np.inf, -np.inf], np.nan)
    x = x.dropna(axis=1, how="all")
    x = x.loc[:, x.nunique(dropna=True) > 1]

    if x.shape[1] < 2:
        return {
            "little_chi2": None,
            "little_df": None,
            "little_pvalue": None,
            "global_mechanism": "NA",
            "reason": "Fewer than two usable numeric features."
        }

    if not x.isna().any().any():
        return {
            "little_chi2": None,
            "little_df": None,
            "little_pvalue": None,
            "global_mechanism": "NA",
            "reason": "No missing values in the tested features."
        }

    a = x.to_numpy(dtype=float)
    observed = np.isfinite(a)
    n, p = a.shape

    # Remove rows with no observed values in the tested features.
    keep = observed.any(axis=1)
    a = a[keep]
    observed = observed[keep]
    n = len(a)

    if n <= p:
        return {
            "little_chi2": None,
            "little_df": None,
            "little_pvalue": None,
            "global_mechanism": "NA",
            "reason": "Insufficient rows relative to feature count."
        }

    col_means = np.nanmean(a, axis=0)
    filled = np.where(observed, a, col_means)
    mu = filled.mean(axis=0)
    cov = np.atleast_2d(np.cov(filled, rowvar=False))
    cov += np.eye(p) * 1e-6

    # EM-style estimation for the multivariate normal mean/covariance.
    converged = False

    for _ in range(max_iter):
        sum_x = np.zeros(p)
        sum_xx = np.zeros((p, p))

        for row, mask in zip(a, observed):
            o = np.where(mask)[0]
            m = np.where(~mask)[0]

            ex = mu.copy()
            ex[o] = row[o]
            exx = np.outer(ex, ex)

            if len(m) > 0 and len(o) > 0:
                soo = cov[np.ix_(o, o)]
                smo = cov[np.ix_(m, o)]
                som = cov[np.ix_(o, m)]
                smm = cov[np.ix_(m, m)]

                inv = np.linalg.pinv(soo)
                ex[m] = (
                    mu[m] + smo @ inv @ (row[o] - mu[o])
                )

                exx = np.outer(ex, ex)
                conditional_cov = smm - smo @ inv @ som
                exx[np.ix_(m, m)] += conditional_cov

            elif len(o) == 0:
                exx = cov + np.outer(mu, mu)

            sum_x += ex
            sum_xx += exx

        new_mu = sum_x / n
        new_cov = sum_xx / n - np.outer(new_mu, new_mu)
        new_cov = (new_cov + new_cov.T) / 2
        new_cov += np.eye(p) * 1e-8

        change = max(
            np.max(np.abs(new_mu - mu)),
            np.max(np.abs(new_cov - cov))
        )

        mu, cov = new_mu, new_cov

        if change < tol:
            converged = True
            break

    # Group rows by distinct observed/missing pattern.
    patterns = {}
    for i, mask in enumerate(observed):
        key = tuple(mask.tolist())
        patterns.setdefault(key, []).append(i)

    statistic = 0.0
    df = -p

    for pattern, indices in patterns.items():
        o = np.where(pattern)[0]
        p_j = len(o)

        # Degrees of freedom count distinct patterns, not rows.
        df += p_j

        if p_j == 0:
            continue

        group = a[np.ix_(indices, o)]
        group_mean = group.mean(axis=0)
        delta = group_mean - mu[o]
        cov_o = cov[np.ix_(o, o)]

        statistic += len(indices) * float(
            delta @ np.linalg.pinv(cov_o) @ delta
        )

    if df <= 0 or not np.isfinite(statistic):
        return {
            "little_chi2": float(statistic),
            "little_df": int(df),
            "little_pvalue": None,
            "global_mechanism": "NA",
            "reason": "Non-positive degrees of freedom or invalid statistic."
        }

    pvalue = float(chi2.sf(statistic, df))

    return {
        "little_chi2": float(statistic),
        "little_df": int(df),
        "little_pvalue": pvalue,
        "global_mechanism": (
            "MCAR-consistent" if pvalue >= MCAR_ALPHA
            else "Not MCAR-consistent"
        ),
        "em_converged": converged,
        "patterns_tested": len(patterns),
        "features_tested": x.columns.tolist()
    }

In [0]:
little_result = littles_mcar_test(pdf, numeric_columns)
# print(little_result)
print(json.dumps(little_result, indent=4))

{
    "little_chi2": 23.789046303377354,
    "little_df": 28,
    "little_pvalue": 0.6926202090020783,
    "global_mechanism": "MCAR-consistent",
    "em_converged": true,
    "patterns_tested": 2,
    "features_tested": [
        "Mazzoni 1- Plodder vaccum pump Start/stop Indication_mean10to22",
        "Mazzoni 1- Vaccum in final plodder hopper_mean10to22",
        "Mazzoni 1 - Billet Temperature at Plodder exit_std10to22",
        "Mazzoni 1 - Flowrate throughput_std10to22",
        "Mazzoni 1- Final Plodder Worm motor current_std10to22",
        "Mazzoni 1- Final plodder Chilled Water flow_std10to22",
        "Mazzoni 1- Final plodder Cone Heater temp_std10to22",
        "Mazzoni 1- Final plodder I/L cooling water temp_std10to22",
        "Mazzoni 1- Final plodder O/L cooling water temp_std10to22",
        "Mazzoni 1- Final plodder Temperature at Cone Inlet_std10to22",
        "Mazzoni 1- Final plodder plate pressure_std10to22",
        "Mazzoni 1- Final plodder worm RPM_std10to22"

**Statistical caution:** This is a custom EM-based implementation and should be validated against a trusted reference implementation before production use. In particular, missingness patterns with very few observations, singular covariance matrices, and EM convergence can affect the test. A non-significant result means consistent with MCAR, not proof of MCAR.

# Test associations with other process features

For each eligible feature, this function tests whether its missingness indicator is associated with other observed features. Numeric predictors use Mann–Whitney U; categorical predictors use chi-square. Benjamini–Hochberg correction is applied across the association tests for each missingness indicator.

In [0]:
def benjamini_hochberg(p_values):
    """
    Benjamini-Hochberg FDR correction.
    Returns adjusted p-values in the original order.
    Uses only Python built-ins.
    """
    n = len(p_values)

    if n == 0:
        return []

    # Keep original positions and sort by p-value
    sorted_indices = sorted(
        range(n),
        key=lambda i: p_values[i]
    )

    adjusted = [1.0] * n
    min_adjusted_p = 1.0

    # Work backwards to ensure adjusted p-values are monotonic
    for rank_index in range(n - 1, -1, -1):
        original_index = sorted_indices[rank_index]
        rank = rank_index + 1
        p_value = p_values[original_index]

        corrected_p = min(1.0, p_value * n / rank)
        min_adjusted_p = min(min_adjusted_p, corrected_p)

        adjusted[original_index] = min_adjusted_p

    return adjusted

In [0]:
def test_missingness_associations(pdf, missing_col):
    missing_indicator = pdf[missing_col].isna().astype(int)
    results = []

    for feature in pdf.columns:
        if feature == missing_col or feature in ID_COLUMNS:
            continue

        x = pdf[feature]
        valid = x.notna() & missing_indicator.notna()
        x_valid = x[valid]
        m_valid = missing_indicator[valid]

        if m_valid.nunique() < 2:
            continue

        try:
            if pd.api.types.is_numeric_dtype(x):
                group1 = x_valid[m_valid == 1].dropna()
                group0 = x_valid[m_valid == 0].dropna()

                if len(group1) == 0 or len(group0) == 0:
                    continue

                u_stat, pvalue = mannwhitneyu(
                    group1, group0, alternative="two-sided"
                )

                effect = (
                    2 * u_stat / (len(group1) * len(group0)) - 1
                )
                test_name = "mann_whitney_u"

            else:
                table = pd.crosstab(x_valid, m_valid)

                if min(table.shape) < 2:
                    continue

                chi2_stat, pvalue, _, _ = chi2_contingency(table)
                n = table.to_numpy().sum()
                denom = min(table.shape[0] - 1, table.shape[1] - 1)

                effect = (
                    float(np.sqrt(chi2_stat / (n * denom)))
                    if n > 0 and denom > 0 else 0.0
                )
                test_name = "chi_square"

            results.append({
                "feature": feature,
                "test": test_name,
                "pvalue": float(pvalue),
                "effect_size": float(effect)
            })

        except (ValueError, TypeError, ZeroDivisionError):
            continue

    if results:
        # adjusted = multipletests(
        #     [r["pvalue"] for r in results],
        #     method="fdr_bh"
        # )[1]
        adjusted = benjamini_hochberg([r["pvalue"] for r in results])

        for result, adjusted_p in zip(results, adjusted):
            result["adjusted_pvalue"] = float(adjusted_p)
            result["associated"] = bool(adjusted_p < 0.05)

    return sorted(
        results,
        key=lambda r: r.get("adjusted_pvalue", 1.0)
    )

In [0]:
pdf

,TrialBusinessId_ of Sigma,Mazzoni 1- Sigma Mixer current_mean,Mazzoni 1- Sigma Mixer current_sum,Mazzoni 1- Sigma Mixer current_median,Mazzoni 1- Sigma Mixer current_std,Mazzoni 1- Plodder vaccum pump Start/stop Indication_mean10to22,Mazzoni 1- Vaccum in final plodder hopper_mean10to22,Mazzoni 1 - Billet Temperature at Plodder exit_std10to22,Mazzoni 1 - Flowrate throughput_std10to22,Mazzoni 1- Final Plodder Worm motor current_std10to22,Mazzoni 1- Final plodder Chilled Water flow_std10to22,Mazzoni 1- Final plodder Chilled water SV on/off indication_std10to22,Mazzoni 1- Final plodder Cone Heater temp_std10to22,Mazzoni 1- Final plodder I/L cooling water temp_std10to22,Mazzoni 1- Final plodder O/L cooling water temp_std10to22,Mazzoni 1- Final plodder Temperature at Cone Inlet_std10to22,Mazzoni 1- Final plodder main motor Start/stop Indication_std10to22,Mazzoni 1- Final plodder plate pressure_std10to22,Mazzoni 1- Final plodder worm RPM_std10to22,Mazzoni 1- Level in Final Plodder hopper_std10to22,Mazzoni 1- Plodder vaccum pump Start/stop Indication_std10to22,Mazzoni 1- Vaccum in final plodder hopper_std10to22,Mazzoni 1 - Billet Temperature at Plodder exit_min10to22,Mazzoni 1 - Flowrate throughput_min10to22,Mazzoni 1- Final Plodder Worm motor current_min10to22,Mazzoni 1- Final plodder Chilled Water flow_min10to22,Mazzoni 1- Final plodder Chilled water SV on/off indication_min10to22,Mazzoni 1- Final plodder Cone Heater temp_min10to22,Mazzoni 1- Final plodder I/L cooling water temp_min10to22,Mazzoni 1- Final plodder O/L cooling water temp_min10to22,Mazzoni 1- Final plodder Temperature at Cone Inlet_min10to22,Mazzoni 1- Final plodder main motor Start/stop Indication_min10to22,Mazzoni 1- Final plodder plate pressure_min10to22,Mazzoni 1- Final plodder worm RPM_min10to22,Mazzoni 1- Level in Final Plodder hopper_min10to22,Mazzoni 1- Plodder vaccum pump Start/stop Indication_min10to22,Mazzoni 1- Vaccum in final plodder hopper_min10to22,Mazzoni 1 - Billet Temperature at Plodder exit_max10to22,Mazzoni 1 - Flowrate throughput_max10to22,Mazzoni 1- Final Plodder Worm motor current_max10to22,Mazzoni 1- Final plodder Chilled Water flow_max10to22,Mazzoni 1- Final plodder Chilled water SV on/off indication_max10to22,Mazzoni 1- Final plodder Cone Heater temp_max10to22,Mazzoni 1- Final plodder I/L cooling water temp_max10to22
0,F-240304-00030,6.913571,1216.788452,6.570513,0.801492,0.904762,-0.448345,4.921438,0.268600,1.743602,1.345090,0.0,2.782832,0.720173,0.871559,1.464220,0.0,4.875995,3.400853,0.0,0.294421,0.133407,24.434362,0.012818,5.60,0.182490,1.0,71.900002,10.643191,19.460865,28.600000,1.0,0.0,10.100000,0.0,0.0,-0.563406,47.459351,1.028169,11.29,5.966179,1.0,82.199997,13.884436
1,F-240304-00033,7.169434,1039.567943,7.111538,0.881639,0.740385,-0.378071,6.613789,0.258907,1.293775,0.731971,0.0,0.874302,0.829211,7.641592,0.640481,0.0,4.741523,2.418906,0.0,0.440547,0.242830,24.435299,0.012818,5.48,0.433414,1.0,81.300003,10.805253,22.126892,30.400000,1.0,0.0,12.000000,0.0,0.0,-0.590323,48.355225,0.965300,10.12,4.593357,1.0,83.900002,14.364008
2,F-240304-00035,7.157731,937.662816,6.607692,1.544077,0.992000,-0.484142,4.740093,0.163025,0.408754,1.703410,0.0,0.622106,0.507022,0.390306,0.590148,0.0,4.519551,2.534354,0.0,0.089443,0.098413,24.436235,0.013123,5.77,0.213596,1.0,81.300003,11.658560,17.531843,33.099998,1.0,0.0,10.900000,0.0,0.0,-0.575983,47.667171,0.811182,7.71,11.749868,1.0,83.400002,13.676069
3,F-240308-00008,7.296046,1729.162822,7.094872,1.062486,0.811594,-0.412801,5.267247,0.307264,1.496816,4.225747,0.0,1.294648,0.507757,0.841943,2.312346,0.0,4.889702,2.515183,0.0,0.391746,0.204901,24.435299,0.013123,0.00,-0.006221,1.0,72.000000,11.367510,18.115036,26.799999,1.0,0.0,10.600000,0.0,0.0,-0.590723,44.374809,1.146886,10.50,16.042532,1.0,77.199997,13.520623
4,F-240308-00009,6.719295,1478.244873,6.344872,0.782194,0.681818,-0.456956,3.963646,0.166160,1.141007,1.205639,0.0,0.536491,0.461895,0.631635,0.487324,0.0,3.739258,2.348854,0.0,0.467099,0.1788

In [0]:
eligible_columns

['Mazzoni 1- Plodder vaccum pump Start/stop Indication_mean10to22',
 'Mazzoni 1- Vaccum in final plodder hopper_mean10to22',
 'Mazzoni 1 - Billet Temperature at Plodder exit_std10to22',
 'Mazzoni 1 - Flowrate throughput_std10to22',
 'Mazzoni 1- Final Plodder Worm motor current_std10to22',
 'Mazzoni 1- Final plodder Chilled Water flow_std10to22',
 'Mazzoni 1- Final plodder Chilled water SV on/off indication_std10to22',
 'Mazzoni 1- Final plodder Cone Heater temp_std10to22',
 'Mazzoni 1- Final plodder I/L cooling water temp_std10to22',
 'Mazzoni 1- Final plodder O/L cooling water temp_std10to22',
 'Mazzoni 1- Final plodder Temperature at Cone Inlet_std10to22',
 'Mazzoni 1- Final plodder main motor Start/stop Indication_std10to22',
 'Mazzoni 1- Final plodder plate pressure_std10to22',
 'Mazzoni 1- Final plodder worm RPM_std10to22',
 'Mazzoni 1- Level in Final Plodder hopper_std10to22',
 'Mazzoni 1- Plodder vaccum pump Start/stop Indication_std10to22',
 'Mazzoni 1- Vaccum in final plodder 

In [0]:
missing_col = "Mazzoni 1- Final Plodder Worm motor current_std10to22"
test_missingness_associations(pdf, missing_col)

[{'feature': 'Mazzoni 1- Sigma Mixer current_sum',
  'test': 'mann_whitney_u',
  'pvalue': 0.015070859122414565,
  'effect_size': -0.5727699530516432,
  'adjusted_pvalue': 0.6028343648965826,
  'associated': False},
 {'feature': 'Mazzoni 1- Sigma Mixer current_mean',
  'test': 'mann_whitney_u',
  'pvalue': 0.6589520595526397,
  'effect_size': -0.11032863849765262,
  'adjusted_pvalue': 1.0,
  'associated': False},
 {'feature': 'Mazzoni 1- Sigma Mixer current_median',
  'test': 'mann_whitney_u',
  'pvalue': 0.48088318457034307,
  'effect_size': -0.17136150234741787,
  'adjusted_pvalue': 1.0,
  'associated': False},
 {'feature': 'Mazzoni 1- Sigma Mixer current_std',
  'test': 'mann_whitney_u',
  'pvalue': 0.12305794264765856,
  'effect_size': -0.3755868544600939,
  'adjusted_pvalue': 1.0,
  'associated': False},
 {'feature': 'Mazzoni 1- Plodder vaccum pump Start/stop Indication_mean10to22',
  'test': 'mann_whitney_u',
  'pvalue': 0.9710129462360889,
  'effect_size': 0.028169014084507005,


For example, if missingness in Mazzoni 1- Final Plodder Worm motor current_std10to22 is significantly associated with the observed Mazzoni 1- Final plodder worm RPM_std10to22, that RPM feature becomes a candidate driving variable. This is evidence of association, not proof of causality.

# Predict missingness using logistic regression

In [0]:
def predict_missingness_auc(pdf, missing_col):
    y = pdf[missing_col].isna().astype(int)

    # Five-fold CV needs at least five examples in each class.
    if y.value_counts().min() < 5:
        return {
            "auc": None,
            "status": "Insufficient class counts for 5-fold CV"
        }

    X = pdf.drop(
        columns=[missing_col] + [
            c for c in ID_COLUMNS if c in pdf.columns
        ],
        errors="ignore"
    ).copy()

    X = X.dropna(axis=1, how="all")
    X = X.loc[:, X.nunique(dropna=True) > 1]

    if X.empty:
        return {"auc": None, "status": "No usable predictors"}

    numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = [
        c for c in X.columns if c not in numeric_cols
    ]

    transformers = []

    if numeric_cols:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_cols
        ))

    if categorical_cols:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_cols
        ))

    model = Pipeline([
        ("preprocessor", ColumnTransformer(transformers)),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ])

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    try:
        scores = cross_val_score(
            model, X, y, cv=cv, scoring="roc_auc"
        )

        return {
            "auc": float(scores.mean()),
            "fold_auc": [float(v) for v in scores],
            "status": "OK; interpret cautiously with small sample"
        }

    except ValueError as exc:
        return {"auc": None, "status": str(exc)}

For your data, most eligible features have only 5 missing rows. Although five-fold cross-validation can technically run with this class count, the AUC will be highly unstable. Treat it as exploratory evidence rather than a reliable basis for classifying the missingness mechanism.

# arget association and final output
Your four target columns are observed in all 148 rows. We will test each target's association with each eligible feature's missingness indicator.

The target association p-value below is the raw p-value for the specific target comparison. The associated-variable list uses adjusted p-values from the multiple-testing correction.

In [0]:
def run_miss02(df, target_columns=TARGET_COLUMNS):
    prepared = prepare_miss02_data(df)

    pdf = prepared["pdf"]
    eligible = prepared["eligible_columns"]
    numeric = prepared["numeric_columns"]

    little = littles_mcar_test(pdf, numeric)
    per_column = []

    for feature in eligible:
        associations = test_missingness_associations(pdf, feature)

        auc_result = predict_missingness_auc(pdf, feature)

        target_results = {
            t: next(
                (
                    a for a in associations
                    if a["feature"] == t
                ),
                None
            )
            for t in target_columns
            if t in pdf.columns and t != feature
        }

        target_assoc_p = {
            t: (r["pvalue"] if r else None)
            for t, r in target_results.items()
        }

        associated_vars = [
            a["feature"] for a in associations
            if a.get("associated", False)
        ][:10]

        significant_target = any(
            p is not None and p < TARGET_ASSOC_ALPHA
            for p in target_assoc_p.values()
        )

        auc = auc_result["auc"]

        if significant_target:
            label = "Informative missingness; investigate possible MNAR"
        elif auc is not None and auc >= MAR_AUC_THRESHOLD:
            label = "MAR evidence"
        elif (
            little["global_mechanism"] == "MCAR-consistent"
            and not associated_vars
        ):
            label = "MCAR-consistent; evidence is limited"
        elif associated_vars:
            label = "Possible MAR; associated variables detected"
        else:
            label = "Inconclusive"

        per_column.append({
            "feature": feature,
            "missing_pct": float(
                prepared["profile"].loc[
                    prepared["profile"]["feature"] == feature,
                    "missing_pct"
                ].iloc[0]
            ),
            "mechanism_label": label,
            "missing_model_auc": auc,
            "auc_status": auc_result["status"],
            "top_associated_vars": associated_vars,
            "target_assoc_p": target_assoc_p,
            "associations": associations
        })

    return {
        "metric_id": "MISS-02",
        "metric_name": "MCAR vs MAR test",
        "sample_rows": prepared["sample_rows"],
        **little,
        "per_column": per_column,
        "status": "NA" if not eligible else "COMPLETED"
    }


miss02_result = run_miss02(df)

print("Global result:")
print({
    k: v for k, v in miss02_result.items()
    if k != "per_column"
})

print("\nPer-column results:")
display(pd.DataFrame(miss02_result["per_column"]))

If no column meets the 1% missingness threshold, the status is NA, as specified in your requirement. If eligible columns exist but the global test cannot be calculated, the global statistics should remain None rather than being treated as evidence for MCAR.

# Expected output structure
The function returns a Python dictionary that you can serialize to JSON or store as a Databricks table.

Output field	Meaning
- little_chi2	: Global Little's MCAR test statistic
- little_df	: Degrees of freedom calculated from distinct patterns
- little_pvalue	: Global MCAR test p-value
- global_mechanism	: MCAR-consistent, Not MCAR-consistent, or NA
- mechanism_label	: Per-feature diagnostic classification
- missing_model_auc	: Mean five-fold cross-validated ROC-AUC, if calculable
- top_associated_vars :	Observed features associated with missingness
- target_assoc_p	: Raw p-value for each configured target association

or example, an illustrative result for one feature could look like this:

```
{
    "feature": "Mazzoni 1- Final Plodder Worm motor current_std10to22",
    "missing_pct": 4.0541,
    "mechanism_label": "Possible MAR; associated variables detected",
    "missing_model_auc": None,
    "top_associated_vars": [
        "Mazzoni 1- Final plodder worm RPM_std10to22"
    ],
    "target_assoc_p": {
        "Mazzoni 1- Sigma Mixer current_mean": None,
        "Mazzoni 1- Sigma Mixer current_sum": None,
        "Mazzoni 1- Sigma Mixer current_median": None,
        "Mazzoni 1- Sigma Mixer current_std": None
    }
}
```

This is a hypothetical example to explain the schema, not the measured result of your DataFrame.

# Important interpretation for your Sigma data
Your dataset has only 148 rows, with 5–6 missing values per eligible feature. This limits the strength of the conclusions.
- If Little's p-value is below 0.05: the global missingness pattern is inconsistent with MCAR under the test assumptions.
- If a feature has AUC ≥ 0.65: observed variables predict its missingness, providing evidence consistent with MAR.
- If a target association has p < 0.01: flag potentially informative missingness, as your requirement specifies.
- If no significant associations are found: do not conclude that MCAR is proven. With such a small sample, the tests may have limited statistical power.
- If missingness is associated with a target: this alone does not prove MNAR. Use the label possible MNAR only when the broader evidence supports that interpretation.

One additional improvement I recommend for your manufacturing dataset is to investigate whether the missing values occur in the same six batch rows across the std, min, and max features. If they do, that pattern may reveal a shared data extraction or process-window issue, which is operationally useful regardless of the statistical classification.